# Registros duplicados del dataset

Este notebook identifica filas exactamente repetidas en el dataset de delivery. La comparacion usa todas las columnas con datos y excluye unicamente la columna `Unnamed` vacia que aparece por la coma final del CSV. No se recortan ni normalizan valores.

Los numeros de fila reportados corresponden al archivo CSV e incluyen el encabezado como fila 1.

In [ ]:
from pathlib import Path
import pandas as pd

ruta_dataset = Path('..') / 'data' / 'online food delivery dataset.csv'
df_original = pd.read_csv(ruta_dataset)
columnas_extra = [columna for columna in df_original.columns if str(columna).startswith('Unnamed')]
df = df_original.drop(columns=columnas_extra).copy()
columnas_comparacion = df.columns.tolist()

print(f'Filas analizadas: {len(df)}')
print(f'Columnas comparadas: {len(columnas_comparacion)}')
print(f'Columnas Unnamed excluidas: {columnas_extra}')

In [ ]:
mascara_duplicados = df.duplicated(subset=columnas_comparacion, keep=False)
duplicados = df.loc[mascara_duplicados].copy()
duplicados['_fila_csv'] = duplicados.index + 2

grupos_duplicados = (
    duplicados.groupby(columnas_comparacion, dropna=False, sort=False)
    .agg(veces=('_fila_csv', 'size'), filas_csv=('_fila_csv', list))
    .reset_index()
)
grupos_duplicados = grupos_duplicados.sort_values(
    ['veces', 'filas_csv'], ascending=[False, True]
).reset_index(drop=True)

numero_grupos = len(grupos_duplicados)
filas_implicadas = int(grupos_duplicados['veces'].sum())
copias_extra = int((grupos_duplicados['veces'] - 1).sum())

print(f'Grupos de registros repetidos: {numero_grupos}')
print(f'Filas que pertenecen a esos grupos: {filas_implicadas}')
print(f'Copias adicionales a la primera aparicion: {copias_extra}')

In [ ]:
resumen_frecuencias = (
    grupos_duplicados.groupby('veces', as_index=False)
    .agg(
        grupos=('veces', 'size'),
        filas_implicadas=('veces', 'sum'),
        copias_extra=('veces', lambda frecuencia: (frecuencia - 1).sum()),
    )
.sort_values('veces', ascending=False)
.reset_index(drop=True)
)
display(resumen_frecuencias)

# La tabla contiene cada registro completo repetido, su frecuencia y las filas del CSV.
display(grupos_duplicados)